# GRPO Text-to-SQL Fine-Tuning on Ray

Two-phase training for text-to-SQL on RHOAI:

1. **SFT Warmup** — LoRA SFT on cleaned BIRD-Platinum + NNDSS data to give the model baseline SQL ability
2. **GRPO RL** — Reinforcement learning with execution-based rewards against SQLite (BIRD) and Trino (NNDSS)

Both phases run as RayJobs on MIG 3g.71gb slices via CodeFlare SDK.

Based on the [ReViSQL](https://thinkingmachines.ai/news/putting-task-expertise-into-rl) approach.

## Setup

In [ ]:
%env UV_EXTRA_INDEX_URL=https://pypi.org/simple
!uv pip install codeflare_sdk

In [ ]:
from codeflare_sdk import ManagedClusterConfig, RayJob
from kubernetes.client import (
    V1PersistentVolumeClaimVolumeSource,
    V1Volume,
    V1VolumeMount,
)

## Authenticate to your OpenShift Cluster

In [ ]:
!oc login --token=<YOUR_TOKEN> --server=<YOUR_API_SERVER> --insecure-skip-tls-verify
!oc whoami
!oc project
# Grant the workbench service account permission to create RayJobs
!oc apply -f ../manifests/rayjob-rbac.yaml

## 1. Configure Training Parameters

In [ ]:
import subprocess

# Cluster configuration
NAMESPACE = subprocess.run(
    ["oc", "project", "-q"], capture_output=True, text=True
).stdout.strip()
IMAGE = "quay.io/modh/ray@sha256:f63a015302758f805e5332605669b886e4d7ac60ec929413a2ffc19a904211c6"  # ray:2.55.1-py312-cu129-th081
HF_TOKEN = ""

# Kueue LocalQueue — set to "" to disable Kueue scheduling
KUEUE_LQ_NAME = "reserved"

# MIG GPU configuration (H200 slices)
GPU_RESOURCE_NAME = "nvidia.com/mig-3g.71gb"  # 71GB VRAM per slice
NUM_WORKERS = 1
N_GPUS_PER_NODE = 1

# Storage
PVC_NAME = "rl-sql-rwx"
PVC_PATH = "shared"
PVC_MOUNT_PATH = f"/opt/app-root/src/{PVC_PATH}"

# Model
MODEL_PATH = "Qwen/Qwen3-8B"

# Phase 1: SFT warmup paths (from notebook 01)
SFT_DATA_PATH = f"{PVC_MOUNT_PATH}/text2sql/sft_train_data.jsonl"
SFT_CKPT_DIR = f"{PVC_MOUNT_PATH}/text2sql/sft_checkpoint"

# Phase 2: GRPO paths
GRPO_DATA_PATH = f"{PVC_MOUNT_PATH}/text2sql/grpo_prompts.jsonl"
GRPO_OUTPUT_DIR = f"{PVC_MOUNT_PATH}/text2sql/grpo_output"

# SFT hyperparameters
SFT_EPOCHS = 2
SFT_MAX_SEQ_LEN = 4096
SFT_LEARNING_RATE = 1e-4

# GRPO hyperparameters (adapted from ReViSQL paper)
NUM_ITERATIONS = 10
TASKS_PER_ITERATION = 16
GROUP_SIZE = 8
N_TRAIN = 2500
N_VAL = 50
GRPO_LEARNING_RATE = 5e-5

# LoRA configuration (rank 32 per ReViSQL)
LORA_R = 32
LORA_ALPHA = 16

# vLLM memory split
GPU_MEMORY_UTILIZATION = 0.20

# Trino connection for NNDSS execution rewards
TRINO_HOST = f"trino.{NAMESPACE}.svc.cluster.local"
TRINO_PORT = 8080

print(f"Namespace:  {NAMESPACE}")
print(f"Kueue LQ:   {KUEUE_LQ_NAME or '(disabled)'}")
print(f"GPU:        {GPU_RESOURCE_NAME}")
print(f"Model:      {MODEL_PATH}")
print(f"Trino:      {TRINO_HOST}:{TRINO_PORT}")

## 2. Configure Shared Storage

In [ ]:
pvc_volume = V1Volume(
    name="training-data",
    persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=PVC_NAME),
)
pvc_mount = V1VolumeMount(name="training-data", mount_path=PVC_MOUNT_PATH)

---
## Phase 1: SFT Warmup

Train a LoRA adapter on the combined BIRD-Platinum + NNDSS SFT data.
This gives the model baseline SQL generation ability so GRPO groups
have meaningful variance in quality.

In [ ]:
env_vars = {}
if HF_TOKEN:
    env_vars["HF_TOKEN"] = HF_TOKEN
    env_vars["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
env_vars["TRINO_HOST"] = TRINO_HOST
env_vars["TRINO_PORT"] = str(TRINO_PORT)

kueue_labels = {"kueue.x-k8s.io/queue-name": KUEUE_LQ_NAME} if KUEUE_LQ_NAME else {}

# Register MIG resource type with CodeFlare SDK
mig_accelerator_configs = {GPU_RESOURCE_NAME: GPU_RESOURCE_NAME}

sft_cluster_config = ManagedClusterConfig(
    image=IMAGE,
    num_workers=0,  # Single GPU SFT on head
    head_cpu_requests=4,
    head_cpu_limits=8,
    head_memory_requests=64,
    head_memory_limits=64,
    head_accelerators={GPU_RESOURCE_NAME: 1},
    volumes=[pvc_volume],
    volume_mounts=[pvc_mount],
    envs=env_vars,
    labels=kueue_labels,
    accelerator_configs=mig_accelerator_configs,
)

print(f"SFT cluster: head-only with 1x {GPU_RESOURCE_NAME}")
print(f"  Kueue: {KUEUE_LQ_NAME or '(disabled)'}") 
print(f"  HF_TOKEN: {'set' if HF_TOKEN else 'not set'}")

In [ ]:
sft_entrypoint = f'''python -c "
import subprocess
from training_hub import lora_sft

lora_sft(
    model_path='{MODEL_PATH}',
    data_path='{SFT_DATA_PATH}',
    ckpt_output_dir='{SFT_CKPT_DIR}',
    num_epochs={SFT_EPOCHS},
    max_seq_len={SFT_MAX_SEQ_LEN},
    learning_rate={SFT_LEARNING_RATE},
    lora_r={LORA_R},
    lora_alpha={LORA_ALPHA},
)

subprocess.run(['chmod', '-R', 'g+r', '{SFT_CKPT_DIR}'], check=True)
print('SFT warmup completed')
"'''

print("SFT entrypoint configured")
print(f"  Model: {MODEL_PATH}")
print(f"  Data:  {SFT_DATA_PATH}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")

In [ ]:
sft_job = RayJob(
    job_name="text2sql-sft-warmup",
    entrypoint=sft_entrypoint,
    cluster_config=sft_cluster_config,
    namespace=NAMESPACE,
    ttl_seconds_after_finished=600,
    local_queue=KUEUE_LQ_NAME or None,
)

# Pin submitter pod to GPU nodes so image is only pulled once
_orig_build = sft_job._build_rayjob_cr
def _patched_build():
    cr = _orig_build()
    tpl = cr["spec"].get("submitterPodTemplate")
    if tpl is None:
        cr["spec"]["submitterPodTemplate"] = {
            "spec": {
                "restartPolicy": "Never",
                "nodeSelector": {"nvidia.com/gpu.present": "true"},
                "containers": [{"name": "ray-job-submitter", "image": IMAGE}],
            }
        }
    else:
        tpl.setdefault("spec", {})["nodeSelector"] = {"nvidia.com/gpu.present": "true"}
        tpl["spec"].setdefault("restartPolicy", "Never")
    return cr
sft_job._build_rayjob_cr = _patched_build

sft_job.submit()
print(f"SFT RayJob '{sft_job.name}' submitted to namespace '{NAMESPACE}'")

In [ ]:
import time

print("Waiting for SFT warmup to complete...")
for i in range(120):
    status, ready = sft_job.status(print_to_console=False)
    print(f"[{i * 30}s] Status: {status.value}")
    if ready or status.value in ("FAILED", "COMPLETE"):
        break
    time.sleep(30)

print(f"\nSFT final status: {status.value}")

In [ ]:
sft_job.logs()

---
## Phase 2: GRPO RL with Execution Rewards

Train with GRPO using the verl backend, starting from the SFT checkpoint.
The reward function executes generated SQL against SQLite (BIRD) and Trino (NNDSS)
and grades results using the ReViSQL grading logic.

In [ ]:
grpo_cluster_config = ManagedClusterConfig(
    image=IMAGE,
    head_cpu_requests=4,
    head_cpu_limits=8,
    head_memory_requests=32,
    head_memory_limits=32,
    num_workers=NUM_WORKERS,
    worker_cpu_requests=8,
    worker_cpu_limits=16,
    worker_memory_requests=128,
    worker_memory_limits=192,
    worker_accelerators={GPU_RESOURCE_NAME: N_GPUS_PER_NODE},
    volumes=[pvc_volume],
    volume_mounts=[pvc_mount],
    envs=env_vars,
    labels=kueue_labels,
    accelerator_configs=mig_accelerator_configs,
)

print(f"GRPO cluster: head (no GPU) + {NUM_WORKERS} worker(s) with {N_GPUS_PER_NODE}x {GPU_RESOURCE_NAME}")
print(f"  Kueue: {KUEUE_LQ_NAME or '(disabled)'}")
print(f"  HF_TOKEN: {'set' if HF_TOKEN else 'not set'}")

In [ ]:
import base64

TOTAL_GPUS = N_GPUS_PER_NODE * NUM_WORKERS

_code = f"""\
import subprocess
subprocess.run(['pip', 'install', 'trino'], check=True)

from training_hub import lora_grpo

result = lora_grpo(
    model_path='{SFT_CKPT_DIR}',
    data_path='{GRPO_DATA_PATH}',
    ckpt_output_dir='{GRPO_OUTPUT_DIR}',
    backend='verl',
    n_gpus={TOTAL_GPUS},
    num_iterations={NUM_ITERATIONS},
    tasks_per_iteration={TASKS_PER_ITERATION},
    group_size={GROUP_SIZE},
    n_train={N_TRAIN},
    n_val={N_VAL},
    lora_r={LORA_R},
    lora_alpha={LORA_ALPHA},
    learning_rate={GRPO_LEARNING_RATE},
    gpu_memory_utilization={GPU_MEMORY_UTILIZATION},
)

subprocess.run(['chmod', '-R', 'g+r', '{GRPO_OUTPUT_DIR}'], check=True)

print('GRPO training complete')
print('Status:', result.get('status'))
print('Reward history:', result.get('reward_history'))
"""

_encoded = base64.b64encode(_code.encode()).decode()
grpo_entrypoint = (
    f"python -c \"import base64; exec(base64.b64decode('{_encoded}').decode())\""
)

print("GRPO entrypoint configured")
print(f"  Starting from SFT checkpoint: {SFT_CKPT_DIR}")
print(f"  GPUs: {TOTAL_GPUS} total")
print(f"  Iterations: {NUM_ITERATIONS}")
print(f"  Group size: {GROUP_SIZE}")

In [ ]:
grpo_job = RayJob(
    job_name="text2sql-grpo-training",
    entrypoint=grpo_entrypoint,
    cluster_config=grpo_cluster_config,
    namespace=NAMESPACE,
    ttl_seconds_after_finished=600,
    local_queue=KUEUE_LQ_NAME or None,
)

# Pin submitter pod to GPU nodes so image is only pulled once
_orig_build_grpo = grpo_job._build_rayjob_cr
def _patched_build_grpo():
    cr = _orig_build_grpo()
    tpl = cr["spec"].get("submitterPodTemplate")
    if tpl is None:
        cr["spec"]["submitterPodTemplate"] = {
            "spec": {
                "restartPolicy": "Never",
                "nodeSelector": {"nvidia.com/gpu.present": "true"},
                "containers": [{"name": "ray-job-submitter", "image": IMAGE}],
            }
        }
    else:
        tpl.setdefault("spec", {})["nodeSelector"] = {"nvidia.com/gpu.present": "true"}
        tpl["spec"].setdefault("restartPolicy", "Never")
    return cr
grpo_job._build_rayjob_cr = _patched_build_grpo

grpo_job.submit()
print(f"GRPO RayJob '{grpo_job.name}' submitted to namespace '{NAMESPACE}'")

In [ ]:
print("Waiting for GRPO training to complete...")
print(f"Expected duration: ~30-90 minutes for {NUM_ITERATIONS} iterations\n")

for i in range(240):
    status, ready = grpo_job.status(print_to_console=False)
    print(f"[{i * 30}s] Status: {status.value}")
    if ready or status.value in ("FAILED", "COMPLETE"):
        break
    time.sleep(30)

print(f"\nGRPO final status: {status.value}")

In [ ]:
grpo_job.logs()

## 3. Plot Reward Curve

In [ ]:
import re

import matplotlib.pyplot as plt

log_output = grpo_job.logs()

rewards = re.findall(r"Reward history:\s*\[([^\]]+)\]", log_output)

if rewards:
    reward_values = [float(r) for r in rewards[-1].split(",")]
    iterations = list(range(1, len(reward_values) + 1))

    plt.figure(figsize=(8, 4))
    plt.plot(iterations, reward_values, marker="o")
    plt.xlabel("Iteration")
    plt.ylabel("Mean Reward")
    plt.title("GRPO Text-to-SQL — Reward Curve")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Final reward: {reward_values[-1]:.4f}")
else:
    print("Could not parse reward history from logs.")

## 4. Cleanup

In [ ]:
# Uncomment to delete jobs immediately:
# sft_job.delete()
# grpo_job.delete()